<a href="https://colab.research.google.com/github/Shreya00320/AI_lab/blob/main/Propositional_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Truth-table enumeration algorithm for deciding propositional entailment (TT-ENTAILS?)

Sentences are written as nested tuples:
    'P'                    -> symbol
    ('not', s)             -> negation          (¬s)
    ('and', s1, s2, ...)   -> conjunction       (s1 ∧ s2 ...)
    ('or',  s1, s2, ...)   -> disjunction       (s1 ∨ s2 ...)
    ('=>',  s1, s2)        -> implication       (s1 → s2)
    ('<=>', s1, s2)        -> biconditional     (s1 ↔ s2)
"""
from itertools import product


# ---------------------------------------------------------------
# PL-TRUE? : does a sentence hold in a model?
# ---------------------------------------------------------------
def pl_true(s, model):
    if isinstance(s, str):
        return model[s]
    op = s[0]
    if op == 'not':
        return not pl_true(s[1], model)
    if op == 'and':
        return all(pl_true(x, model) for x in s[1:])
    if op == 'or':
        return any(pl_true(x, model) for x in s[1:])
    if op == '=>':
        return (not pl_true(s[1], model)) or pl_true(s[2], model)
    if op == '<=>':
        return pl_true(s[1], model) == pl_true(s[2], model)
    raise ValueError(f"Unknown operator: {op}")


def to_str(s):
    """Pretty print a sentence."""
    if isinstance(s, str):
        return s
    op = s[0]
    if op == 'not':
        return "¬" + to_str(s[1])
    sym = {'and': ' ∧ ', 'or': ' ∨ ', '=>': ' → ', '<=>': ' ↔ '}[op]
    return "(" + sym.join(to_str(x) for x in s[1:]) + ")"


def symbols_in(s, found=None):
    """List of proposition symbols in a sentence (in order of appearance)."""
    if found is None:
        found = []
    if isinstance(s, str):
        if s not in found:
            found.append(s)
    else:
        for x in s[1:]:
            symbols_in(x, found)
    return found


# ---------------------------------------------------------------
# TT-ENTAILS?  (exactly as in the algorithm on the slide)
# ---------------------------------------------------------------
def tt_entails(kb, alpha):
    """kb : list of sentences (their conjunction is the KB)
       alpha : query sentence"""
    symbols = []
    for s in kb + [alpha]:
        symbols_in(s, symbols)
    return tt_check_all(kb, alpha, symbols, {})


def tt_check_all(kb, alpha, symbols, model):
    if not symbols:                                   # EMPTY?(symbols)
        if all(pl_true(s, model) for s in kb):        # PL-TRUE?(KB, model)
            return pl_true(alpha, model)              # then return PL-TRUE?(alpha, model)
        return True                                   # KB false -> always true
    p, rest = symbols[0], symbols[1:]
    return (tt_check_all(kb, alpha, rest, {**model, p: True}) and
            tt_check_all(kb, alpha, rest, {**model, p: False}))


# ---------------------------------------------------------------
# Truth table printer (for the observation book)
# ---------------------------------------------------------------
def tf(b):
    return "True" if b else "False"


def print_truth_table(kb, alpha=None):
    symbols = []
    for s in kb + ([alpha] if alpha is not None else []):
        symbols_in(s, symbols)

    cols = symbols + [to_str(s) for s in kb] + ["KB"]
    if alpha is not None:
        cols.append(f"α = {to_str(alpha)}")
    width = [max(len(c), 5) for c in cols]

    line = " | ".join(c.center(w) for c, w in zip(cols, width))
    print(line)
    print("-" * len(line))

    kb_true_models = []
    for values in product([False, True], repeat=len(symbols)):
        model = dict(zip(symbols, values))
        row = [tf(v) for v in values]
        sentence_vals = [pl_true(s, model) for s in kb]
        kb_val = all(sentence_vals)
        row += [tf(v) for v in sentence_vals] + [tf(kb_val)]
        if alpha is not None:
            row.append(tf(pl_true(alpha, model)))
        mark = "   <-- KB is true" if kb_val else ""
        print(" | ".join(r.center(w) for r, w in zip(row, width)) + mark)
        if kb_val:
            kb_true_models.append(model)

    print("\nModels in which KB is true:")
    if not kb_true_models:
        print("   (none - KB is unsatisfiable)")
    for m in kb_true_models:
        print("   " + ", ".join(f"{k}={tf(v)}" for k, v in m.items()))
    return kb_true_models


def check(kb, alpha):
    result = tt_entails(kb, alpha)
    print(f"Does KB entail {to_str(alpha)} ?  ->  {'YES' if result else 'NO'}")
    return result


# ---------------------------------------------------------------
# OBSERVATION BOOK PROBLEM
#   KB:  Q -> P,   P -> ¬Q,   Q ∨ R
# ---------------------------------------------------------------
if __name__ == "__main__":
    kb = [
        ('=>', 'Q', 'P'),            # Q → P
        ('=>', 'P', ('not', 'Q')),   # P → ¬Q
        ('or', 'Q', 'R'),            # Q ∨ R
    ]

    print("=========== (i) Truth table for the KB ===========")
    print_truth_table(kb)

    print("\n=========== (ii) Does KB entail R ? ===========")
    print_truth_table(kb, 'R')
    check(kb, 'R')

    print("\n=========== (iii) Does KB entail R → P ? ===========")
    print_truth_table(kb, ('=>', 'R', 'P'))
    check(kb, ('=>', 'R', 'P'))

    print("\n=========== (iv) Does KB entail Q → R ? ===========")
    print_truth_table(kb, ('=>', 'Q', 'R'))
    check(kb, ('=>', 'Q', 'R'))

    # ---- Example from the slides: KB = (A ∨ C) ∧ (B ∨ ¬C), α = A ∨ B ----
    print("\n\n=========== Slide example: KB = (A∨C) ∧ (B∨¬C), α = A∨B ===========")
    kb2 = [('or', 'A', 'C'), ('or', 'B', ('not', 'C'))]
    alpha2 = ('or', 'A', 'B')
    print_truth_table(kb2, alpha2)
    check(kb2, alpha2)

=========== (i) Truth table for the KB ===========
  Q   |   P   |   R   | (Q → P) | (P → ¬Q) | (Q ∨ R) |   KB 
------------------------------------------------------------
False | False | False |   True  |   True   |  False  | False
False | False |  True |   True  |   True   |   True  |  True   <-- KB is true
False |  True | False |   True  |   True   |  False  | False
False |  True |  True |   True  |   True   |   True  |  True   <-- KB is true
 True | False | False |  False  |   True   |   True  | False
 True | False |  True |  False  |   True   |   True  | False
 True |  True | False |   True  |  False   |   True  | False
 True |  True |  True |   True  |  False   |   True  | False

Models in which KB is true:
   Q=False, P=False, R=True
   Q=False, P=True, R=True

=========== (ii) Does KB entail R ? ===========
  Q   |   P   |   R   | (Q → P) | (P → ¬Q) | (Q ∨ R) |   KB  | α = R
--------------------------------------------------------------------
False | False | False |   True  | 